# Chương 2 · Thực nghiệm và đánh giá

Notebook trình bày kết quả 6 thực nghiệm của Chương 2: bảng số, biểu đồ và nhận xét cho từng thực nghiệm.

- **Số liệu** lấy từ các CSV trong `results/`, do `scripts/run_benchmark_grid.jl` đo (môi trường đo ở mục 0).
  Notebook **không chạy lại thực nghiệm**, chỉ đọc kết quả để vẽ.
- **Code vẽ** nằm trong `scripts/make_figures.jl`; notebook `include` file đó rồi gọi từng hàm.
  Mỗi hàm cũng lưu hình (`.pdf`, `.png`) vào `results/figures/` và bảng LaTeX vào
  `docs/IEEE-conference-template-062824/tables/` để chèn vào báo cáo.

**Cách chạy:** mở notebook với kernel Julia rồi *Run all*. Cell đầu tự kích hoạt môi trường vẽ `scripts/plots`
(chỉ có Plots.jl); lần đầu cài Plots mất vài phút.

**Quy ước trên mọi biểu đồ:** nhóm màu xanh dương, điểm tròn; SPMF màu cam, điểm vuông. Thời gian là trung vị của
3 lần đo và **gồm đọc và ghi file** ở cả hai phía (SPMF: "Total time" tự in), để so công bằng.

## 0 · Thiết lập và môi trường đo

In [ ]:
using Pkg
# Thư mục gốc repo: nơi có src/FPClose.jl và scripts/make_figures.jl
function find_repo(dir = pwd())
    while true
        isfile(joinpath(dir, "scripts", "make_figures.jl")) && return dir
        parent = dirname(dir)
        parent == dir && error("Không tìm thấy repo FPClose: mở notebook từ trong repo")
        dir = parent
    end
end
REPO_DIR = find_repo()
Pkg.activate(joinpath(REPO_DIR, "scripts", "plots"); io = devnull)
Pkg.instantiate()
include(joinpath(REPO_DIR, "scripts", "make_figures.jl"))

show_md(s) = display("text/markdown", s)
show_md(read(joinpath(REPO_DIR, "results", "ENVIRONMENT.md"), String))

## 1 · Correctness

**Đề:** với mỗi tập dữ liệu, so số lượng frequent itemset và support của từng itemset giữa cài đặt của nhóm và SPMF;
báo cáo tỉ lệ itemset khớp hoàn toàn; nếu có sai lệch thì phân tích nguyên nhân.

**Cách đo:** 56 cấu hình (9 dataset × 5–7 mức minsup). So **closed itemsets** của `fpclose` với SPMF `FPClose`, và
**mọi frequent itemset** (recovery của nhóm) với SPMF `FPGrowth_itemsets` ở mức minsup cao nhất của mỗi dataset.

Tỉ lệ khớp = itemset khớp (cùng itemset, cùng support) / số itemset khác nhau của cả hai phía, cộng dồn mọi cấu hình
(nguồn: `results/correctness/grid_correctness.csv`).

In [ ]:
show_md(correctness_table())

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Tỉ lệ khớp trên closed và trên mọi frequent itemset là bao nhiêu?
- Vì sao mọi frequent itemset chỉ so ở minsup cao nhất (so số closed và số frequent itemset, ví dụ mushrooms 20%)?
- Không có sai lệch: nêu các lỗi đã gặp và sửa trong quá trình làm (xem `docs/DEVIATIONS.md`: item 0 trùng nút gốc, làm tròn minsup, ...).

## 2 · Thời gian chạy theo minsup

**Đề:** với mỗi tập dữ liệu, vẽ thời gian chạy theo minsup giảm dần (5–7 điểm), so đường cong của nhóm và SPMF trên
cùng đồ thị.

**Cách đo:** `results/runtime/grid_runtime_summary.csv`, cột `time_s`. Trục x là các mức minsup đã chọn, cách đều,
giảm dần từ trái sang phải; trục y thang log.

### Hình tổng hợp (dùng trong báo cáo)

In [ ]:
runtime_figure()

### Tỉ lệ thời gian nhóm / SPMF

Nhỏ hơn 1 nghĩa là nhóm nhanh hơn SPMF.

In [ ]:
show_md(runtime_ratio_table())

### Từng dataset

Cùng số liệu với hình tổng hợp, vẽ riêng để xem kỹ.

In [ ]:
for d in DATASETS
    display(runtime_figure(d))
end

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Trên dữ liệu dày (chess, mushrooms) và thưa (retail, T10, kosarak), nhóm nhanh hay chậm hơn SPMF, bao nhiêu lần?
- Khoảng cách thay đổi thế nào khi minsup giảm (connect đổi chiều ở 75%)?
- Giải thích dựa trên bài báo: chi phí dựng FP-tree và cây điều kiện, kiểm tra đóng trên CFI-tree; đọc/ghi file chiếm bao nhiêu (cột `read_s`, `write_s` trong `grid_runtime.csv`)?

## 3 · Số lượng itemset theo minsup

**Đề:** vẽ số lượng itemset sinh ra theo minsup; nhận xét quan hệ giữa minsup và output size, so sánh dữ liệu dày và thưa.

**Cách đo:** số closed itemsets ở mỗi cấu hình (đã khớp SPMF 100% ở mục 1). Hai trục đều thang log; trục x là
minsup thật (%) nên các dataset có khoảng minsup khác nhau nằm ở vị trí khác nhau.

In [ ]:
itemsets_figure()

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Số itemset tăng nhanh cỡ nào khi minsup giảm, ở dữ liệu dày so với dữ liệu thưa?
- Vì sao dữ liệu dày cần minsup cao (60–95%) còn dữ liệu thưa dùng minsup rất thấp (0.02–1%)?
- Liên hệ output size với thời gian chạy ở mục 2.

## 4 · Bộ nhớ: bản cơ bản và bản tối ưu

**Đề:** đo và báo cáo RAM tối đa (peak memory) cho mỗi tập dữ liệu tại minsup trung bình; so sánh bản cơ bản và bản
tối ưu của nhóm.

**Cách đo:** `results/memory/basic_vs_optimized.csv`. Mỗi biến thể chạy 3 lần trong tiến trình Julia riêng; lấy trung vị.
- **basic:** tắt hết tối ưu. **pruning-only:** chỉ bật cắt nhánh dòng 8 của Fig. 8. **optimized:** bật hết
  (cắt nhánh, FP-array, early termination, exact count, CFI-tree rút gọn).
- `peak RSS`: bộ nhớ đỉnh của tiến trình, gồm khoảng 285 MB nền của Julia. `Tăng thêm`: peak − nền, phần do dữ liệu
  và khai thác. `Cấp phát`: tổng bộ nhớ cấp phát trong bước đọc file và khai thác.
- Chỉ so các biến thể của nhóm với nhau: "Max memory" của SPMF là heap JVM, đo khác RSS.

In [ ]:
memory_figure()

In [ ]:
tables = memory_tables()
show_md(tables.memory)

### Ablation: tắt từng tối ưu

Thời gian khai thác của từng biến thể chia cho bản tối ưu (lớn hơn 1: chậm hơn bản tối ưu).

In [ ]:
show_md(tables.ablation)

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Bản tối ưu tiết kiệm bao nhiêu bộ nhớ và thời gian so với bản cơ bản? Có dataset nào tối ưu lại tốn hơn (kosarak)?
- Tối ưu nào đóng góp nhiều nhất, tối ưu nào gần như không có tác dụng (bảng ablation)? Vì sao (dữ liệu dày/thưa, điều kiện dùng FP-array)?
- Vì sao tổng cấp phát lớn hơn nhiều so với phần bộ nhớ tăng thêm (accidents, kosarak: khoảng 3.4 GB cấp phát)?

## 5 · Khả năng mở rộng (scalability)

**Đề:** chọn một tập dữ liệu lớn (Retail hoặc Accidents), tạo tập con 10%, 25%, 50%, 75%, 100% giao dịch; vẽ thời gian
chạy theo kích thước CSDL; nhận xét xu hướng tuyến tính hay phi tuyến.

**Cách đo:** `data/scalability/retail_p010..p100.txt` (tập con lồng nhau, seed 2026), hai mức minsup 0.1% và 0.05%.
Hình (b) chia thời gian cho thời gian ở 100%: nằm trên đường chéo nghĩa là tăng tuyến tính theo số giao dịch.

In [ ]:
sc = scalability_figure_and_table()
display(sc.figure)
show_md(sc.table)

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Thời gian của nhóm và SPMF tăng tuyến tính hay phi tuyến theo số giao dịch?
- Vì sao SPMF nằm trên đường chéo (chi phí cố định lớn ở kích thước nhỏ) còn nhóm nằm gần hoặc dưới?
- Ở kích thước nào nhóm bắt đầu chậm hơn SPMF, và vì sao?

## 6 · Ảnh hưởng của độ dài giao dịch trung bình

**Đề:** nếu thuật toán bị ảnh hưởng bởi độ dài giao dịch, thiết kế thực nghiệm minh chứng bằng CSDL tổng hợp có độ dài
giao dịch tăng dần.

**Cách đo:** `data/synthetic/len05..len30.txt` (10 000 giao dịch, 500 item, độ dài trung bình 5 đến 30; giao dịch dài
chứa giao dịch ngắn), hai mức minsup 0.5% và 0.2%.

In [ ]:
tx = txlen_figure_and_table()
display(tx.figure)
show_md(tx.table)

### Nhận xét

_TODO: viết nhận xét. Gợi ý các câu cần trả lời:_

- Thời gian và số closed itemsets tăng thế nào khi giao dịch dài hơn (gần như theo hàm mũ?)
- Giải thích bằng cấu trúc FP-tree: giao dịch dài làm cây sâu hơn, nhiều cây điều kiện hơn, CFI-tree lớn hơn.
- Nhóm so với SPMF thay đổi thế nào theo độ dài?

## 7 · Đánh giá chung và hướng tối ưu

_TODO_
- **Điểm mạnh** của cài đặt so với SPMF (loại dữ liệu nào, vì sao).
- **Điểm yếu** (loại dữ liệu nào, nguyên nhân: đọc/ghi file, cấp phát bộ nhớ, ...).
- **Ít nhất 2 hướng tối ưu cụ thể** có thể làm tiếp, kèm số liệu ở trên làm căn cứ (tham khảo `docs/DEVIATIONS.md`).